# LC 703 — Kth Largest Element in a Stream
**Difficulty:** Easy &nbsp;|&nbsp; **Category:** Heap
**Pattern:** Min-Heap of Size K — Running Kth Largest

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Keep a min-heap of
exactly k elements. The smallest value in that
heap is always the kth largest overall — everything
below it was already evicted.
</div>

## Official Problem Statement

Design a class to find the `k`th largest element
in a stream. Note that it is the `k`th largest
element in the sorted order, not the `k`th
distinct element.

Implement `KthLargest` class:

- `KthLargest(int k, int[] nums)` Initialises the
  object with the integer `k` and the stream of
  integers `nums`.
- `int add(int val)` Appends the integer `val` to
  the stream and returns the element representing
  the `k`th largest element in the stream.

**Example 1:**
```
Input:
  ["KthLargest","add","add","add","add","add"]
  [[3,[4,5,8,2]],[3],[5],[10],[9],[4]]
Output:
  [null, 4, 5, 5, 8, 8]
```

**Constraints:**
- `1 <= k <= 10^4`
- `0 <= nums.length <= 10^4`
- `-10^4 <= nums[i] <= 10^4`
- `-10^4 <= val <= 10^4`
- At most `10^4` calls to `add`
- It is guaranteed that there will always be at
  least `k` elements in the array when `add` is
  called

## What This Is Actually Asking

Numbers keep arriving one at a time in a stream.
After each new number arrives, report what the
kth largest number is across everything seen so
far. For k=3, report the 3rd largest after every
addition. Do it without re-sorting every time.

## Walk Through an Example by Hand

```
k=3   nums=[4,5,8,2]

Init — build heap from [4,5,8,2], keep top 3:
  Push 4  heap=[4]
  Push 5  heap=[4,5]
  Push 8  heap=[4,5,8]
  Push 2  heap=[4,5,8,2]  size=4 > k=3
          pop min(2) -> heap=[4,5,8]  size=3
  Heap now = [4, 5, 8]   heap[0]=4 is 3rd largest

add(3):
  Push 3  heap=[3,5,8,4]  size=4 > 3
  pop min(3)              heap=[4,5,8]
  return heap[0] = 4

add(5):
  Push 5  heap=[4,5,8,5]  size=4 > 3
  pop min(4)              heap=[5,5,8]
  return heap[0] = 5

add(10):
  Push 10 heap=[5,5,8,10] size=4 > 3
  pop min(5)              heap=[5,8,10]
  return heap[0] = 5

add(9):
  Push 9  heap=[5,8,10,9] size=4 > 3
  pop min(5)              heap=[8,9,10]
  return heap[0] = 8

add(4):
  Push 4  heap=[4,9,10,8] size=4 > 3
  pop min(4)              heap=[8,9,10]
  return heap[0] = 8
```

## The Picture

```
All numbers seen so far (k=3):
  [2, 3, 4, 4, 5, 5, 8, 9, 10]
   ^^^^^^^^^^^^^^^^^  ^^^^^^^^
   everything smaller  top 3
   was evicted         kept in heap

Min-heap of size k = a WINDOW on the top k:

  heap = [8, 9, 10]
          ^
          smallest in top-k = kth largest overall

Why min-heap (not max)?
  Max-heap keeps the biggest at top —
  but we need the BOTTOM of the top-k,
  which is the minimum of the top-k bucket.
  A min-heap surfaces that in O(1).

When a new number arrives:
  push it                    O(log k)
  if heap size > k: pop min  O(log k)
  return heap[0]             O(1)
```

## When To Use This Pattern

- When you need the **kth largest in a stream**,
  think **min-heap of size k**
- When the heap top is too small for a new value,
  think **push then pop min to maintain size k**
- When you need to answer after every insertion,
  think **heap[0] is the answer in O(1)**
- When k is fixed and the stream grows, think
  **O(log k) per add, not O(n log n) per sort**

## The Approach

In `__init__`, store k and build a min-heap from
the initial list. Trim it down to size k by popping
the minimum until only k elements remain.
In `add`, push the new value onto the heap. If the
heap now has more than k elements, pop the minimum.
Return the heap's top element — that is the kth
largest.

In [ ]:
import heapq  # Python's min-heap (heapify, heappush, heappop)

In [ ]:
def test_harness(cls):
    """
    Replay sequences of (op, args, expected) against cls.
    ops: 'init' -> (k, nums),  expected ignored
         'add' -> (val,),      expected is the return value
    """
    sequences = [
        # sequence 1 — from the problem statement
        [
            ("init", (3, [4,5,8,2]),  None),
            ("add",  (3,),             4),
            ("add",  (5,),             5),
            ("add",  (10,),            5),
            ("add",  (9,),             8),
            ("add",  (4,),             8),
        ],
        # sequence 2 — k=1, always return max
        [
            ("init", (1, []),          None),
            ("add",  (1,),             1),
            ("add",  (3,),             3),
            ("add",  (2,),             3),
        ],
        # sequence 3 — k equals stream size
        [
            ("init", (3, [1,2,3]),     None),
            ("add",  (0,),             1),
            ("add",  (4,),             2),
        ],
    ]

    passed = 0
    for s_i, seq in enumerate(sequences):
        obj = None
        seq_pass = True
        for op, args, expected in seq:
            if op == "init":
                obj = cls(*args)
            else:
                result = obj.add(*args)
                ok = result == expected
                if not ok:
                    seq_pass = False
                    print(
                        f"  Seq {s_i+1} FAILED: "
                        f"add{args} expected "
                        f"{expected} got {result}"
                    )
        if seq_pass:
            passed += 1
            print(f"Sequence {s_i+1}: PASSED")
        else:
            print(f"Sequence {s_i+1}: FAILED")

    print(f"\n{passed}/{len(sequences)} sequences passed")

In [ ]:
class KthLargest:
    """
    Returns kth largest element after each stream add.

    Store a min-heap of exactly k elements. Init by
    heapifying nums and trimming to k. On add: push
    value; if size > k pop the minimum. heap[0] is
    always the kth largest seen so far.

    add:  O(log k) — push + conditional pop
    Space: O(k) — heap holds at most k elements
    """

    def __init__(self, k: int, nums):
        pass

    def add(self, val: int) -> int:
        pass


# Quick debug — run this cell while building
kl = KthLargest(3, [4,5,8,2])
print(kl.add(3))   # 4
print(kl.add(5))   # 5
print(kl.add(10))  # 5
print(kl.add(9))   # 8

In [ ]:
# Uncomment and run when solution is ready
# test_harness(KthLargest)

## Complexity

| Approach | Time per add | Space |
|---|---|---|
| Re-sort every add | O(n log n) | O(n) |
| Min-heap of size k | O(log k) | O(k) |

The heap approach is faster because it only
maintains the k largest elements — it never looks
at anything that has already been evicted.

## Real World Connection

At Citi, the telemetry pipeline receives CPU
readings from 6,000 servers in a continuous stream.
The on-call dashboard must always show the top-K
most overloaded servers without re-ranking all
6,000 entries every second.
A min-heap of size K tracks the K heaviest servers
in O(log K) per incoming reading — the same pattern
as this problem.
On AWS Lambda, the same heap approach sizes the
concurrency reservation: after each invocation
metric arrives, the heap immediately surfaces the
kth highest burst without a full re-sort.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra